In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from fenics import *

# Create mesh and define function spaces
mesh = UnitSquareMesh(32, 32)  # 2D unit square mesh
V = VectorFunctionSpace(mesh, 'P', 2)  # Velocity function space (P2 elements)
Q = FunctionSpace(mesh, 'P', 1)  # Pressure function space (P1 elements)

# Define mixed function space using MixedElement
element = MixedElement([V.ufl_element(), Q.ufl_element()])  # Combine velocity and pressure elements
W = FunctionSpace(mesh, element)  # Mixed function space for velocity and pressure

# Define boundary conditions for velocity
u_inlet = Expression(('1.0 - x[1]*x[1]', '0.0'), degree=2)  # Inlet velocity profile
bc_u_inlet = DirichletBC(W.sub(0), u_inlet, 'on_boundary && near(x[0], 0)')  # Inlet condition

# Define trial and test functions for the mixed space
(u, p) = TrialFunctions(W)  # Velocity and pressure trial functions
(v, q) = TestFunctions(W)   # Velocity and pressure test functions

# Define parameters for Navier-Stokes equation
nu = 0.001  # Kinematic viscosity

# Incompressible Navier-Stokes equations (Mixed Formulation)
# Define the bilinear forms for the mixed formulation

# First weak form (momentum equation with pressure)
F1 = (inner(1.0/nu * grad(u), grad(v)) - div(v) * p - div(u) * q) * dx

# Second weak form (incompressibility condition)
F2 = inner(grad(p), grad(q)) * dx

# Variational problem (left-hand sides and right-hand sides)
a = lhs(F1 + F2)
L = rhs(F1 + F2)

# Mixed formulation - combined velocity and pressure function
w = Function(W)  # Combined function space for velocity and pressure

# Define the mixed problem for the Navier-Stokes equations
problem = LinearVariationalProblem(a, L, w, bc_u_inlet)

# Use the mixed solver
solver = LinearVariationalSolver(problem)
solver.solve()

# Extract the velocity and pressure solutions
u_sol, p_sol = w.split()

# Visualize the results
plot(u_sol, title="Velocity field")
plt.show()

plot(p_sol, title="Pressure field")
plt.show()